In [3]:
import pandas as pd

df = pd.read_csv("/data2/yuyao/methane_emission/data_csv/CM_s2_l8_s2_time_train.csv")
print(df["label"].value_counts())  # 看是不是全是 0 或全是 1

print("positives:", (df["label"]==1).sum())
print("negatives:", (df["label"]==0).sum())


label
0    21657
1    21656
Name: count, dtype: int64
positives: 21656
negatives: 21657


In [1]:

import pandas as pd
for path in ['../data_csv/CM_s2_l8_s2.csv']:
    df = pd.read_csv(path)
    print(path, len(df), df['label'].mean(), (df['plume_pixels_in_crop']==0).mean(), df['time_offset_hours'].max())


../data_csv/CM_s2_l8_s2.csv 48126 0.5 0.5 47.99804888888889


In [ ]:
import tifffile as tiff

path = "../CM_s2_l8_s2/ang20160915t180022-B/s2_20160916T184422Z/bg_01/image.tif"

img = tiff.imread(path)
print(img.shape, img.dtype)

(12, 32, 32) uint16


In [2]:
import pandas as pd

def balance_csv(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    # 分开两类
    df0 = df[df["label"] == 0]
    df1 = df[df["label"] == 1]

    # 按 1 类数量下采样 0 类
    df0_down = df0.sample(n=len(df1), random_state=42)

    # 合并 & shuffle
    df_balanced = pd.concat([df0_down, df1], axis=0).sample(frac=1, random_state=42)

    df_balanced.to_csv(output_csv, index=False)

    print(f"Original: class0={len(df0)}, class1={len(df1)}")
    print(f"Balanced: class0={len(df0_down)}, class1={len(df1)}")
    print(f"Saved to: {output_csv}")


# 使用
balance_csv(
    "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout/CM_s2_l8_s2_time_train.csv",
    "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout/CM_s2_l8_s2_time_train_balanced.csv"
)

balance_csv(
    "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout/CM_s2_l8_s2_time_test.csv",
    "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout/CM_s2_l8_s2_time_test_balanced.csv"
)

Original: class0=22178, class1=11152
Balanced: class0=11152, class1=11152
Saved to: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout/CM_s2_l8_s2_time_train_balanced.csv
Original: class0=5563, class1=2770
Balanced: class0=2770, class1=2770
Saved to: /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/train_tryout/CM_s2_l8_s2_time_test_balanced.csv


In [1]:
import pandas as pd

# Display options — expand columns & rows
pd.set_option("display.max_columns", None)  # show all columns
pd.set_option("display.max_rows", 200)      # adjust if necessary
pd.set_option("display.width", 2000)        # wide console formatting
pd.set_option("display.max_colwidth", None) # don't truncate long strings

path = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/hongxuan/methane/carbonmapper_3_90_365_l2a_test_data_classification_temporal_96/train.csv"
df = pd.read_csv(path, low_memory=False)  # removes dtype warning

print(df.shape)
print(df.dtypes.value_counts())

# Numeric summaries
num_stats = df.describe(include="number")
print(num_stats)

# Categorical summaries (top values, counts, etc.)
cat_stats = df.describe(include="object")
print(cat_stats)

# Optional: missing values
missing_counts = df.isna().sum().sort_values(ascending=False)
print(missing_counts.head(20))


(85785, 10)
object     4
float64    4
int64      2
Name: count, dtype: int64
                 id         label  emission_auto  emission_uncertainty_auto      latitude     longitude
count  85785.000000  85785.000000   81070.000000               81087.000000  85785.000000  85785.000000
mean   42893.000000      0.495658     307.649033                  77.074225     33.696833    -92.020069
std    24764.140758      0.499984     863.795694                 187.583221      8.139826     42.065029
min        1.000000      0.000000       0.000000                   0.000000    -38.031401   -123.383712
25%    21447.000000      0.000000       0.000000                   0.000000     31.927888   -107.701675
50%    42893.000000      0.000000       0.000000                   0.000000     32.267128   -103.011520
75%    64339.000000      1.000000     256.269309                  81.444221     36.671319   -101.811257
max    85785.000000      1.000000   26390.402054                3382.213589     53.893987  

In [1]:
# use MethaneS2CM for training
import pandas as pd
from pathlib import Path

src_dir = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/hongxuan/methane/carbonmapper_3_90_365_l2a_test_data_classification_temporal_32")
dst_dir = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/hongxuan/methane/carbonmapper_3_90_365_l2a_test_data_classification_temporal_32")
dst_dir.mkdir(parents=True, exist_ok=True)

def build_subset(split):
    df = pd.read_csv(src_dir / f"{split}.csv")
    df = df.assign(
        label=df["label"],
        category=df["label"],
        image_path = df["s2_path"].str.extract(r"carbonmapper_3_90_365_l2a_test_data_classification_temporal_32/(\d.*)")[0],
        id=df["id"],
    )[["label", "category", "image_path", "id"]]
    df.to_csv(dst_dir / f"new_{split}.csv", index=False)

for split in ("train", "test"):
    build_subset(split)


In [ ]:
import pandas as pd
import rasterio
from pyproj import Transformer
import os
import numpy as np
from datetime import datetime

def detect_shifts_from_processed_folder(csv_path, processed_base_dir):
    df = pd.read_csv(csv_path)
    
    shifted_count = 0
    centered_count = 0
    missing_count = 0
    
    results = []

    print(f"开始检测 {len(df)} 条数据...")
    print(f"搜索目录: {processed_base_dir}")

    for index, row in df.iterrows():
        # 1. 检查这一行是否有下载记录
        if row.get('has_same_day_s2') == 0:
            continue
            
        plume_id = row.get('plume_id')
        dt_str = row.get('s2_1_datetime') # 例如: 2016-09-06T18:29:22+00:00
        
        if pd.isna(dt_str):
            continue

        # 2. 构造文件名
        # Script 1 的命名规则是: s2_YYYYMMDDTHHMMSSZ.tif
        try:
            # 解析时间字符串
            dt = datetime.fromisoformat(dt_str)
            # 转换为 UTC 并格式化为文件名时间戳
            tif_stamp = dt.strftime("%Y%m%dT%H%M%SZ")
            filename = f"s2_{tif_stamp}.tif"
        except Exception as e:
            # print(f"时间格式解析错误: {dt_str}")
            continue

        # 3. 构造完整路径
        # 路径格式: /.../carbonmapper_data_s2_l2a/{plume_id}/{filename}
        tif_path = os.path.join(processed_base_dir, str(plume_id), filename)
        
        # 4. 检查文件是否存在
        if not os.path.exists(tif_path):
            # 尝试找一下目录下有没有唯一的tif，防止时间戳有细微差异
            # (这是一个容错步骤，可选)
            plume_dir = os.path.join(processed_base_dir, str(plume_id))
            if os.path.exists(plume_dir):
                files = [f for f in os.listdir(plume_dir) if f.startswith("s2_") and f.endswith(".tif")]
                if len(files) > 0:
                    tif_path = os.path.join(plume_dir, files[0]) # 拿第一个凑合用
                else:
                    missing_count += 1
                    continue
            else:
                missing_count += 1
                continue

        try:
            with rasterio.open(tif_path) as src:
                # 5. 获取羽流坐标
                plume_lat = row.get('plume_latitude')
                plume_lon = row.get('plume_longitude')

                # 6. 转换坐标并计算距离
                transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
                plume_x, plume_y = transformer.transform(plume_lon, plume_lat)

                img_center_x = (src.bounds.left + src.bounds.right) / 2
                img_center_y = (src.bounds.top + src.bounds.bottom) / 2

                diff_x = abs(plume_x - img_center_x)
                diff_y = abs(plume_y - img_center_y)
                distance = np.sqrt(diff_x**2 + diff_y**2)

                # 7. 判断平移 (阈值 15米)
                if distance > 15.0:
                    status = "SHIFTED"
                    shifted_count += 1
                else:
                    status = "CENTERED"
                    centered_count += 1
                
                results.append({
                    'plume_id': plume_id,
                    'offset_meters': round(distance, 2),
                    'status': status
                })

        except Exception as e:
            missing_count += 1

    print("-" * 30)
    print(f"检测完成！")
    print(f"有效检测文件: {len(results)}")
    print(f"正中心 (Centered): {centered_count}")
    print(f"发生偏移 (Shifted): {shifted_count}  <-- 这些是 Script 1 拯救的数据")
    print(f"未找到文件: {missing_count}")
    
    return pd.DataFrame(results)

# --- 请修改这里 ---
# 1. 你的 CSV 路径
csv_file = "/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file_with_s2.csv"

# 2. 你的处理结果文件夹路径 (就是截图里展示的那个)
# 请确保这个路径在你的新服务器上是真实存在的
processed_dir = "/data2/yuyao/methane_emission/carbonmapper_data_s2_l2a"

# 运行
df_result = detect_shifts_from_processed_folder(csv_file, processed_dir)

/tmp/ipykernel_2158257/3496649024.py:9: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)
/home/yuyao/miniconda3/envs/data/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


开始检测 24470 条数据...
搜索目录: /data2/yuyao/methane_emission/carbonmapper_data_s2_l2a
------------------------------
检测完成！
有效检测文件: 0
正中心 (Centered): 0
发生偏移 (Shifted): 0  <-- 这些是 Script 1 拯救的数据
未找到文件: 2903
